# Module 5: Statistical Inference in Practice

So far, we have learned how to:

- Work with probability distributions
- Understand sampling and the Central Limit Theorem
- Estimate population parameters
- Build confidence intervals
- Formulate hypotheses
- Perform statistical tests
- Select an appropriate test for a problem

Before applying these ideas to a complete dataset, there are a few practical considerations that help us interpret statistical results correctly.

This module focuses on one question:

> How do we move from a statistical result to a sensible real-world decision?

The workflow is:

$$
\boxed{
\text{Test Result}
\rightarrow
\text{Check Reliability}
\rightarrow
\text{Measure Importance}
\rightarrow
\text{Interpret Carefully}
\rightarrow
\text{Make a Decision}
}
$$

# 1. Statistical Significance vs Practical Significance

Suppose an e-commerce company tests two versions of its website.

The average order values are:

$$
\text{Version A}=₹2000
$$

$$
\text{Version B}=₹2005
$$

With a very large dataset, this small difference may become statistically significant.

But the business must still ask:

> Is an increase of ₹5 per order important enough to justify changing the website?

These are two different questions.

### Statistical Significance

Asks:

> Is the observed difference unlikely to be explained by random sampling variation?

### Practical Significance

Asks:

> Is the size of the difference large enough to matter in the real world?

Therefore:

$$
\boxed{
\text{Statistically Significant}
\neq
\text{Practically Important}
}
$$

A p-value helps us evaluate evidence against the Null Hypothesis.

It does not tell us whether the effect is large or valuable.

## Quick Case: Two Marketing Campaigns

Suppose two campaigns produce different average customer spending.

We will test whether the difference is statistically significant and then examine how large the difference actually is.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

campaign_a = np.random.normal(
    loc=2000,
    scale=300,
    size=5000
)

campaign_b = np.random.normal(
    loc=2015,
    scale=300,
    size=5000
)

print("Campaign A Mean:", round(campaign_a.mean(), 2))
print("Campaign B Mean:", round(campaign_b.mean(), 2))
print(
    "Difference:",
    round(campaign_b.mean() - campaign_a.mean(), 2)
)

In [ ]:
statistic, p_value = stats.ttest_ind(
    campaign_a,
    campaign_b,
    equal_var=False
)

print("P-Value:", round(p_value, 5))

if p_value < 0.05:
    print("The difference is statistically significant.")
else:
    print("The difference is not statistically significant.")

The p-value answers whether the observed difference provides statistical evidence.

The business question is different:

> Is the actual difference in spending large enough to matter?

This is why statistical results should always be interpreted together with the magnitude of the effect.

# 2. Effect Size

Hypothesis testing asks:

> Is there evidence of a difference?

Effect size asks:

> How large is the difference?

One common measure for comparing two means is **Cohen's $d$**.

$$
d=
\frac{\bar{x}_1-\bar{x}_2}
{s_{\text{pooled}}}
$$

A rough interpretation is:

| Cohen's $d$ | Interpretation |
|---|---|
| Around 0.2 | Small effect |
| Around 0.5 | Medium effect |
| Around 0.8 or more | Large effect |

These values are guidelines, not universal rules. The business context remains important.

In [ ]:
mean_difference = (
    campaign_b.mean() -
    campaign_a.mean()
)

pooled_std = np.sqrt(
    (
        campaign_a.var(ddof=1) +
        campaign_b.var(ddof=1)
    ) / 2
)

cohens_d = mean_difference / pooled_std

print("Mean Difference:", round(mean_difference, 2))
print("Cohen's d:", round(cohens_d, 3))

### Interpretation

A test may produce a small p-value while the effect size remains small.

This can happen when the sample size is very large.

A better analysis therefore considers both:

$$
\boxed{
\text{Statistical Significance}
+
\text{Effect Size}
+
\text{Business Context}
}
$$

# 3. Type I and Type II Errors

Whenever we make a decision using hypothesis testing, there is a possibility of making an incorrect decision.

Consider a fraud detection problem.

Our hypotheses could be simplified as:

$$
H_0:
\text{Transaction is genuine}
$$

$$
H_1:
\text{Transaction is fraudulent}
$$

Two important errors are possible.

### Type I Error

We reject a true Null Hypothesis.

A genuine transaction is flagged as fraud.

This is a **false positive**.

### Type II Error

We fail to reject a false Null Hypothesis.

A fraudulent transaction is treated as genuine.

This is a **false negative**.

## The Error Matrix

| Reality | Decision | Result |
|---|---|---|
| $H_0$ is true | Fail to reject $H_0$ | Correct decision |
| $H_0$ is true | Reject $H_0$ | Type I Error |
| $H_0$ is false | Reject $H_0$ | Correct decision |
| $H_0$ is false | Fail to reject $H_0$ | Type II Error |

The probability of a Type I Error is controlled by:

$$
\alpha
$$

The probability of a Type II Error is:

$$
\beta
$$

The importance of these errors depends on the problem.

For example:

- Fraud detection
- Medical diagnosis
- Credit approval
- Manufacturing quality control

may assign very different costs to false positives and false negatives.

# 4. Statistical Power

Statistical power is the probability that a test correctly detects an effect when a real effect exists.

$$
\text{Power}=1-\beta
$$

Higher power means a greater ability to detect a genuine difference.

Power generally increases when:

- Sample size increases
- Effect size increases
- Variability decreases

The practical idea is simple:

> A very small sample may fail to detect a real effect.

This does not necessarily mean that the effect does not exist.

It may mean that the study does not contain enough information to detect it reliably.

## Quick Simulation: Sample Size and Detection

Suppose two customer groups have slightly different average spending.

We will repeatedly sample from the groups using different sample sizes and observe how often the test detects the difference.

In [ ]:
def detection_rate(sample_size, repetitions=500):

    significant_results = 0

    for i in range(repetitions):

        group_a = np.random.normal(
            loc=2000,
            scale=300,
            size=sample_size
        )

        group_b = np.random.normal(
            loc=2100,
            scale=300,
            size=sample_size
        )

        _, p_value = stats.ttest_ind(
            group_a,
            group_b,
            equal_var=False
        )

        if p_value < 0.05:
            significant_results += 1

    return significant_results / repetitions


for n in [20, 50, 100, 300]:

    rate = detection_rate(n)

    print(
        "Sample Size:",
        n,
        "| Detection Rate:",
        round(rate, 3)
    )

### Observation

As the sample size increases, the test is more likely to detect the real difference between the groups.

This connects directly with what we learned earlier:

$$
n \uparrow
\Rightarrow
SE \downarrow
\Rightarrow
\text{More Precise Estimation}
$$

More precise estimates generally make real effects easier to detect.

However, larger samples can also make very small effects statistically significant.

This is another reason to examine effect size and practical importance.

# 5. Assumptions Before Running a Test

Statistical tests are based on certain assumptions.

We do not need to turn every analysis into a long checklist, but a few basic checks are useful.

For tests involving numerical means, common considerations include:

- Are observations independent?
- Are there extreme outliers?
- Is the distribution reasonably suitable for the chosen test?
- Are group variances similar when the test assumes equal variance?

A practical workflow is:

$$
\boxed{
\text{Visualize}
\rightarrow
\text{Check Basic Assumptions}
\rightarrow
\text{Choose Test}
\rightarrow
\text{Interpret}
}
$$

## Quick Check 1: Look at the Distribution

Visual inspection is often a useful starting point.

In [ ]:
sample_data = np.random.normal(
    loc=100,
    scale=15,
    size=100
)

plt.figure(figsize=(8, 4))
plt.hist(
    sample_data,
    bins=15,
    edgecolor="black"
)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.title("Distribution of Sample Data")
plt.show()

## Quick Check 2: Normality Test

One commonly used test is the Shapiro-Wilk test.

The hypotheses are:

$$
H_0:
\text{Data is consistent with a Normal distribution}
$$

$$
H_1:
\text{Data is not consistent with a Normal distribution}
$$

In [ ]:
statistic, p_value = stats.shapiro(sample_data)

print("P-Value:", round(p_value, 4))

if p_value < 0.05:
    print("Evidence suggests departure from Normality.")
else:
    print("No strong evidence against Normality.")

### Important Note

A normality test should not be used blindly.

With very large samples, even small departures from Normality may become statistically significant.

Visual inspection, sample size, outliers, and the robustness of the selected test should also be considered.

# 6. Parametric vs Non-Parametric Tests

The tests used earlier, such as T-Tests and ANOVA, are commonly described as **parametric tests**.

They make assumptions about the underlying population or distribution.

When these assumptions are seriously violated, a non-parametric alternative may sometimes be appropriate.

A practical reference is:

| Situation | Parametric Test | Common Non-Parametric Alternative |
|---|---|---|
| Two independent groups | Independent T-Test | Mann-Whitney U Test |
| Same group measured twice | Paired T-Test | Wilcoxon Signed-Rank Test |
| Three or more independent groups | One-Way ANOVA | Kruskal-Wallis Test |

The purpose here is not to learn another complete family of tests.

The main idea is:

> If the assumptions of a parametric test are seriously unsuitable, check whether a non-parametric alternative is more appropriate.

## Quick Case: Mann-Whitney U Test

Suppose customer spending is highly skewed and we want to compare two independent customer groups.

In [ ]:
group_a = np.array([
    500, 550, 600, 650, 700,
    750, 800, 900, 1200, 5000
])

group_b = np.array([
    400, 450, 500, 550, 600,
    650, 700, 750, 800, 900
])

statistic, p_value = stats.mannwhitneyu(
    group_a,
    group_b,
    alternative="two-sided"
)

print("Mann-Whitney U Statistic:", statistic)
print("P-Value:", round(p_value, 4))

The interpretation of the p-value follows the same general decision framework:

$$
p<0.05
\Rightarrow
\text{Reject }H_0
$$

The statistical test changes, but the reasoning process remains familiar.

# 7. Correlation Does Not Mean Causation

Suppose we observe that two variables move together.

For example:

$$
\text{Advertising Spend}
\uparrow
$$

and:

$$
\text{Sales}
\uparrow
$$

This may indicate an association.

But it does not automatically prove:

$$
\text{Advertising caused the increase in sales}
$$

Other variables may also influence both.

For example:

- Seasonality
- Discounts
- Product launches
- Economic conditions

Statistical association can be useful for prediction.

Causal conclusions require stronger study designs and additional assumptions.

Therefore:

$$
\boxed{
\text{Association}
\neq
\text{Causation}
}
$$

# 8. The Multiple Testing Problem

Suppose a dataset contains 100 features.

We perform 100 separate hypothesis tests using:

$$
\alpha=0.05
$$

Even when no real relationships exist, some tests may appear significant purely by chance.

This is known as the **multiple testing problem**.

The more tests we perform, the greater the chance of finding false positives.

In larger statistical studies, methods such as:

- Bonferroni Correction
- False Discovery Rate

can be used to control this problem.

For our workflow, the main lesson is:

> Do not treat every p-value below 0.05 as an important discovery when testing a large number of variables.

# 9. A Better Statistical Decision Framework

Statistical analysis should not end with:

```python
if p_value < 0.05:
    print("Significant")
```

A more complete workflow is:

### Step 1: Define the Question

What are we trying to understand?

### Step 2: Identify the Variables

Are they numerical or categorical?

### Step 3: Choose the Appropriate Test

Use the structure of the problem.

### Step 4: Check Important Assumptions

Look at the data before running the test.

### Step 5: Perform the Test

Calculate the test statistic and p-value.

### Step 6: Make the Statistical Decision

Compare:

$$
p
$$

with:

$$
\alpha
$$

### Step 7: Examine the Effect Size

How large is the observed difference or relationship?

### Step 8: Consider Practical Importance

Does the result matter in the real-world context?

### Step 9: Communicate the Conclusion

Translate the statistical result into clear language.

# 10. Final Consolidation Case

A company introduces a new customer engagement strategy.

After implementation, average monthly spending increases from approximately:

$$
₹2000
$$

to:

$$
₹2050
$$

A statistical test produces:

$$
p=0.01
$$

Before concluding that the strategy is successful, consider the following questions:

1. Is the result statistically significant?
2. How large is the actual increase?
3. Is the increase practically valuable to the business?
4. Was the sample size extremely large?
5. Were the customer groups comparable?
6. Could another factor explain the increase?
7. Does the effect remain consistent over time?

This is the difference between simply running a statistical test and performing statistical analysis.

# 11. From Module 1 to Module 5

The complete statistical journey is now:

$$
\boxed{
\text{Probability}
\rightarrow
\text{Random Variables}
\rightarrow
\text{Distributions}
\rightarrow
\text{Sampling}
\rightarrow
\text{CLT}
\rightarrow
\text{Estimation}
\rightarrow
\text{Confidence Intervals}
\rightarrow
\text{Hypothesis Testing}
\rightarrow
\text{Statistical Decision}
}
$$

Each stage answers a different question.

| Stage | Main Question |
|---|---|
| Probability | How uncertain is an event? |
| Distribution | How does a random variable behave? |
| Sampling | How can a sample represent a population? |
| CLT | How do sample means behave? |
| Estimation | What is the likely population value? |
| Confidence Interval | How uncertain is our estimate? |
| Hypothesis Testing | Is the observed evidence inconsistent with a claim? |
| Effect Size | How large is the difference? |
| Statistical Decision | Does the result matter in context? |

# Module Summary

The purpose of statistical inference is not simply to calculate p-values.

A good statistical conclusion considers several pieces of evidence together:

$$
\boxed{
\text{P-Value}
+
\text{Effect Size}
+
\text{Assumptions}
+
\text{Sample Size}
+
\text{Business Context}
}
$$

The main lessons from this module are:

- Statistical significance does not automatically imply practical importance.
- Effect size helps describe the magnitude of a difference.
- Type I and Type II errors represent different kinds of incorrect decisions.
- Statistical power describes our ability to detect a real effect.
- Sample size affects both precision and statistical power.
- Statistical tests rely on assumptions that should be considered.
- Non-parametric tests provide alternatives in some situations.
- Association does not automatically imply causation.
- Running many hypothesis tests increases the risk of false discoveries.

We are now ready to move from isolated statistical examples to a complete dataset.

The next stage is:

$$
\boxed{
\text{Real Dataset}
\rightarrow
\text{EDA}
\rightarrow
\text{Statistical Questions}
\rightarrow
\text{Hypothesis Testing}
\rightarrow
\text{Predictive Insights}
}
$$